# Alpaca LIVE account (read-only)

Shows your **live** trading account and saves your holdings for the dashboard alert.

- **Look:** account summary, positions, recent and open orders, market clock (US Central) and the last month's equity.
- **Save:** `my_positions.csv` (Symbol,Shares; the top alert in the app and `holdings_alert.py` use it) and a snapshot in
   `Reports/live_portfolio_snapshot.csv` + `Reports/live_account_history.csv`

**Safety:** only `https://api.alpaca.markets/v2` is accepted (anything else is refused), only read-only GET requests are
made, and nothing in this notebook or in `alpaca_paper.py` can place, change or cancel an order. Keys are never printed.

**Setup (once):** add these two lines to `.env` in the project folder (values from Alpaca → Trading → API keys):
```
ALPACA_LIVE_KEY_ID=...
ALPACA_LIVE_SECRET_KEY=...
```
Same from the command line: `python alpaca_paper.py` (print) or `python alpaca_paper.py --sync` (print + save);
`python run_all.py --sync-live` refreshes `my_positions.csv` as part of the daily run.

In [ ]:
import pandas as pd

import alpaca_paper

WRITE_FILES = True   # step 7 writes my_positions.csv and the Reports snapshot files
try:
    acct = alpaca_paper.PaperAccount()          # refuses any URL other than the live endpoint
    print("Connected to", acct.base_url)
except alpaca_paper.PaperAccountError as e:
    acct = None
    print("Not connected:", e)
    print("Add ALPACA_LIVE_KEY_ID and ALPACA_LIVE_SECRET_KEY to .env, then run the notebook again.")

### Account, positions, orders and equity (read-only)

In [ ]:
if acct:
    summary = acct.account_summary()
    display(pd.Series(summary, name="Live account").to_frame())

if acct:
    positions = acct.positions()
    display(positions.round(2))
    print(f"{len(positions)} positions, market value ${positions['Market value'].sum():,.2f}, "
          f"unrealized P/L ${positions['Unrealized P/L'].sum():,.2f}")

if acct:
    display(acct.recent_orders(limit=20))

if acct:
    clock = acct.market_clock()
    print("Market:", "OPEN" if clock["Is open"] else "CLOSED",
          "| next open (CT):", clock["Next open (CT)"] or "n/a",
          "| next close (CT):", clock["Next close (CT)"] or "n/a")
    print("\nOpen orders:")
    display(acct.open_orders(limit=20))
    print("Equity history (daily, last month):")
    display(acct.portfolio_history())

### Save `my_positions.csv` and the portfolio snapshot

In [ ]:
if acct and WRITE_FILES:
    result = alpaca_paper.sync_paper_account(acct, positions_csv=alpaca_paper.POSITIONS_CSV,
                                             snapshot_csv=alpaca_paper.SNAPSHOT_CSV, history_csv=alpaca_paper.HISTORY_CSV)
    print(f"Saved {result['Positions']} positions to {result['positions_csv']}")
    print("Snapshot:", alpaca_paper.SNAPSHOT_CSV, "| history:", alpaca_paper.HISTORY_CSV)
elif acct:
    print("WRITE_FILES is False - nothing saved.")